# Session 13 · Case study: error analysis of 20 misclassified decisions

**Block 3 practice.** Analyse 20 misclassified decisions with the help of their English keywords and the English heading texts, look at the most informative n-grams, and use what you find to improve the classifier.

Run time: about 5 minutes.

Theory: [03-error-analysis-and-limits.md](../theory/03-error-analysis-and-limits.md).

*Author: course team, licence CC-BY-4.0.*

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
decisions = pd.read_parquet(DATA / "train_sample.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet").set_index("heading")
texts = decisions["description"]                     # one document = one description of goods
print(decisions.shape, decisions["heading"].nunique(), "headings")

In [ ]:
year = decisions["start_date"].dt.year
train, valid = decisions[year <= 2021], decisions[year >= 2022]   # validation by time, as on the leaderboard
print(len(train), "training decisions (2017-2021),", len(valid), "validation decisions (2022-2023)")

In [ ]:
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline


def make_model(**tfidf_kw):
    kw = dict(min_df=2, sublinear_tf=True) | tfidf_kw
    return make_pipeline(TfidfVectorizer(**kw),
                         SGDClassifier(loss="hinge", alpha=1e-5, max_iter=20, tol=None, random_state=0, n_jobs=-1))


model = make_model().fit(train["description"], train["heading"])
pred = model.predict(valid["description"])
print("accuracy", round(accuracy_score(valid["heading"], pred), 3),
      "macro-F1", round(f1_score(valid["heading"], pred, average="macro"), 3))

## 1. Where are the errors?

In [ ]:
val = valid.assign(pred=pred)
wrong = val[val["heading"] != val["pred"]]
print(len(wrong), "errors; share inside the right chapter:",
      round((wrong["heading"].str[:2] == wrong["pred"].str[:2]).mean(), 2))
top_pairs = wrong[["heading", "pred"]].value_counts().head(10).rename("n").reset_index()
top_pairs["true_name"] = top_pairs["heading"].map(nomenclature["heading_description"]).str[:45]
top_pairs["pred_name"] = top_pairs["pred"].map(nomenclature["heading_description"]).str[:45]
top_pairs

**Question.** Headings 8524 and 8548 are confused most often. Look up 8524 in the nomenclature: when was it created? (Hint: HS 2022.)

## 2. Draw 20 errors and categorise them

The seed is fixed so that everyone in the class reads the same decisions. Read each one with its keywords and the two heading texts.

In [ ]:
pd.set_option("display.max_colwidth", 250)
errors20 = wrong.sample(20, random_state=1).copy()
errors20["true_heading"] = errors20["heading"] + " " + errors20["heading"].map(nomenclature["heading_description"]).str[:60]
errors20["predicted"] = errors20["pred"] + " " + errors20["pred"].map(nomenclature["heading_description"]).str[:60]
errors20["text"] = errors20["description"].str.replace(r"\s+", " ", regex=True).str[:250]
errors20[["language", "true_heading", "predicted", "keywords", "text"]]

Give each decision **one** category. Suggested categories (add your own):

- `borderline`: related headings; the legal notes decide
- `material-function`: the model follows the material, the tariff the function (or the reverse)
- `language`: rare language or few training examples
- `technical`: chemical or technical knowledge needed
- `set`: sets and composite goods
- `new-heading`: heading created or changed in HS 2022
- `unclear`: no clear reason

In [ ]:
# fill in one category per decision, in the order of the table above
my_categories = [""] * 20
errors20["category"] = my_categories
errors20["category"].value_counts()

## 3. What does the model rely on?

In [ ]:
vec = model.named_steps["tfidfvectorizer"]
clf = model.named_steps["sgdclassifier"]
names = vec.get_feature_names_out()
top = {h: list(names[np.argsort(clf.coef_[list(clf.classes_).index(h)])[::-1][:12]])
       for h in ["6403", "6404", "9503", "3926", "8517"]}
pd.DataFrame(top)

**Question.** Several of the top features are numbers such as `6403`. Count how often a description quotes its own heading, and how accurate the model is with and without such a number.

In [ ]:
has_number = pd.Series([bool(re.search(rf"(?<!\d){h}(?!\d)", t))
                        for h, t in zip(valid["heading"], valid["description"])], index=valid.index)
print("share quoting the heading:", round(has_number.mean(), 3))
val.groupby(has_number).apply(lambda g: pd.Series({"n": len(g), "accuracy": (g["heading"] == g["pred"]).mean()})).round(3)

## 4. Improve the classifier

Evaluate every change on the validation years, and report the subgroup without a quoted heading number separately: that is where the model has to understand the description.

In [ ]:
no_number = ~has_number.to_numpy()


def evaluate(name, **tfidf_kw):
    p = make_model(**tfidf_kw).fit(train["description"], train["heading"]).predict(valid["description"])
    return {"change": name, "accuracy": accuracy_score(valid["heading"], p),
            "macro_F1": f1_score(valid["heading"], p, average="macro"),
            "accuracy without number": accuracy_score(valid["heading"][no_number], p[no_number])}


results = [evaluate("baseline (word unigrams)"),
           evaluate("min_df=1", min_df=1)]
pd.DataFrame(results).round(3)

## Your turn

1. Pick the largest *fixable* category from your 20 decisions and design one change for it (examples: character n-grams with `analyzer="char_wb", ngram_range=(3, 5), max_features=300_000` (about 2 minutes), word bigrams, a different `alpha`, `class_weight="balanced"`, or training on more data). Add it to `results`.
2. Is the improvement larger than the noise? Bootstrap the validation accuracy of the best two settings (Session 7) and compare the intervals.
3. If a change helps, refit it on all sample decisions and submit (see workbook 05).
4. Write three sentences for your team log: the most common error category, the change you tried, and its effect with and without quoted numbers.

In [ ]:
# your code here